# Device safety

In PyTorch, mixing devices is a runtime crash:

```python
a = torch.randn(4, device="mps"); b = torch.randn(4)   # b on cpu
a + b
# RuntimeError: Expected all tensors to be on the same device, found mps:0 and cpu!
```

In idris-ml the executor (backend) is a **0-quantity phantom parameter** on `Tensor`
(`Tensor dims (0 ex : Executor) dt g`), erased at runtime. The compiler catches device
mismatches, the build's linkage table makes un-built backends unspellable, and `toExecutor`
is the one explicit bridge between them. (This notebook runs on a `tape` build, where only
`TapeExecutor` is linked — so the device-mismatch demos here show up as **linkage** errors,
which is exactly the "I targeted a backend that isn't there" case.)

## A tensor on the build's executor

`TapeExecutor` is linked in this build, so constructing on it succeeds:

In [ ]:
:exec do { a <- tconstScalar {ex=TapeExecutor} {dt=F64} 2.0;
  b <- tconstScalar {ex=TapeExecutor} {dt=F64} 3.0;
  c <- tadd a b;
  putStrLn ("2 + 3 = " ++ show (tensorItem c)) }

## "CUDA on a Mac": the linkage gate

`Linked ex` is an empty marker whose instances are generated per build from `BACKEND`.
Every tensor constructor carries `Linked ex =>`, so naming a backend this build *didn't*
compile in is a **compile error** — not a runtime `AssertionError: Torch not compiled with
CUDA enabled`. The next cell is *expected to fail* (a `tape` build has no
`TorchExecutor`):

In [ ]:
:exec do { x <- tconstScalar {ex=TorchExecutor (TCuda 0)} {dt=F64} 0.0;
  putStrLn "should not reach here" }

## Metal is F32-only: the `Compatible` gate

A second, *build-independent* gate: `Compatible (0 ex) (0 dt)` enumerates admissible
(device, dtype) pairs. There is deliberately no `Compatible (MlxExecutor MGpu) F64` (mlx
0.31 dropped float64 on Metal), so asking for it is a compile error — PyTorch's runtime
`TypeError: Cannot convert a MPS Tensor to float64 …` lifted to the type system. We isolate
the dtype axis with a tiny witness that needs only `Compatible` (no construction, so no
`Linked`):

In [ ]:
compatOK : Compatible ex dt => ()
compatOK = ()

In [ ]:
:exec putStrLn (show (compatOK {ex=MlxExecutor MGpu} {dt=F64}))

## `toExecutor`: the one explicit bridge

Moving a tensor between backends is never implicit — it's an explicit `toExecutor`, the only operation that changes the executor type:

In [ ]:
:t toExecutor

## Multi-backend in one program

Because `Executor` is an *open* kind, a multi-link build (`BACKEND=tape,torch,mlx`) links
all three backends into one dylib, and a single program can hold tensors on different
executors at once — the compiler tracking each, `toExecutor` bridging them. No mainstream
framework offers this. See [Why idris-ml §3](../../../../docs/users/why-idris-ml.md) for the
worked `Tape → Torch → Mlx → Tape` round-trip.

## Zero runtime cost

The executor parameter is quantity `0` — fully erased before code generation. A
`Tensor [4] TapeExecutor F64 g` and a `Tensor [4] (TorchExecutor TCpu) F64 g` have the
exact same runtime representation; the safety is purely at compile time, like Rust's
lifetime annotations.

## Summary

| | PyTorch | idris-ml |
|---|---------|----------|
| Device tracking | runtime (`.device`) | compile-time (phantom `ex`) |
| Mismatch detection | `RuntimeError` in the forward pass | type error before the program runs |
| Un-built backend | `AssertionError` at runtime | `Linked` — unspellable at compile time |
| Metal float64 | `TypeError` at op launch | missing `Compatible` instance |
| Device transfer | `.to(device)` | `toExecutor` (explicit, type-changing) |
| Multi-backend in one program | n/a (one runtime) | yes — open `Executor` kind + `toExecutor` |

Next: [08 Hyperparameter Optimization](08_hpo.ipynb) · [09 Precision and Devices](09_precision_devices.ipynb).